# Lab: comparing basal friction laws
```{note}
This lab accompanies {doc}`basal-motion` and feeds into the grounding-line flux discussion in {doc}`../cryosphere/ice-sheets`. Run it in the Firedrake / icepack Docker environment with the **Firedrake (icepack)** kernel.
```
The three friction laws introduced in {doc}`basal-motion` look superficially similar — each produces a basal shear stress that resists sliding — but they make profoundly different predictions when the ice thins near the grounding line. This lab makes that difference concrete on a synthetic glacier inspired by Thwaites. You will
- set up a 2-D flowline domain with a retrograde bed and effective pressure tied to subglacial water connectivity,
- implement all three friction laws as custom functionals and feed each into icepack's SSA / hybrid solver,
- calibrate the laws so they agree at a single point — the calibration trap — and then see them disagree everywhere else,
- apply a 20 m thinning perturbation at the terminus and compare the speed-up each law predicts, and
- reason about which law you would trust to project future Thwaites behaviour.
The central lesson is not which law is correct. It is that the choice of friction law is a first-order modelling decision, and that the two laws most in use today — the Weertman power law and the regularized Coulomb — can be made to agree in the present and still diverge by factors of two or more in the future.

## Background: effective pressure and the Thwaites setting
Thwaites Glacier in West Antarctica sits on a bed that deepens inland — a retrograde slope — so the bed beneath its ice stream lies hundreds of metres below sea level. Warm Circumpolar Deep Water intrudes beneath the floating ice shelf at the terminus, melts the base, and thins the ice near the grounding line. Because of the retrograde bed, thinning at the grounding line reduces the effective pressure there, which matters for the last two friction laws in our hierarchy.
The effective pressure is
$$
N(x) = p_i(x) - p_w(x),
$$
the ice overburden minus the subglacial water pressure. In the **full connectivity** assumption — the simplest drainage model and the one most commonly used in prognostic ice-sheet models — the subglacial water system is everywhere connected to the ocean, so the water pressure equals the ocean pressure at the bed:
$$
p_w(x) = -\rho_w g\, b(x) \quad \text{(where } b < 0\text{)}.
$$
The ice overburden is $p_i = \rho_i g H$, so
$$
N(x) = \rho_i g H(x) - \rho_w g |b(x)|.
$$
Near the grounding line, where the ice is near flotation, $N \to 0$. That limit is where the Weertman law and the Coulomb law diverge most sharply.

In [ ]:
import firedrake
import icepack
import numpy as np
import matplotlib.pyplot as plt
from firedrake import Constant, Function, as_vector, sqrt, inner, conditional, ge
# Physical constants
rho_i = 917.0          # ice density, kg/m^3
rho_w = 1028.0         # ocean water density, kg/m^3
g     = 9.81           # gravitational acceleration, m/s^2
# Domain: a 1-D flowline represented as a narrow rectangle.
# x runs from 0 (inland) to L (grounding zone / terminus).
L  = 200e3    # flowline length, m
Ly = 1e3      # transverse width (fictitious for 1-D), m
nx, ny = 200, 2
mesh = firedrake.RectangleMesh(nx, ny, L, Ly)
Q = firedrake.FunctionSpace(mesh, "CG", 2)    # scalar fields
V = firedrake.VectorFunctionSpace(mesh, "CG", 2)   # velocity
x, y = firedrake.SpatialCoordinate(mesh)
print(f"Domain: {L/1e3:.0f} km × {Ly/1e3:.0f} km,  mesh {nx}×{ny}")

## The synthetic geometry
We want a geometry that captures the essence of a marine outlet glacier: a bed that deepens inland (retrograde), a surface that roughly follows hydrostatic equilibrium, and ice thick enough to be grounded throughout the domain.
**Bed.** The retrograde profile is
$$
b(x) = -100 - 600\,\frac{x}{L},
$$
so the bed goes from $-100$ m at the seaward end ($x = L$) to $-700$ m at the inland end ($x = 0$). There is a gentle sill near the terminus, which is where the grounding line would naturally sit.
**Thickness.** We prescribe a smooth initial thickness of
$$
H(x) = 1000 - 300\,\frac{x}{L}.
$$
The surface is $s = b + H$, which we compute and check for flotation.
**Effective pressure.** Under full connectivity,
$$
N(x) = \rho_i g H(x) + \rho_w g\, b(x) \quad (b < 0),
$$
clipped to a small positive minimum so the friction laws stay well-defined everywhere in the domain.

In [ ]:
# --- Bed, thickness, surface, effective pressure ---
b_expr = -100.0 - 600.0 * x / L         # retrograde bed (m), deeper inland
H_expr =  1000.0 - 300.0 * x / L        # thickness (m)
s_expr = b_expr + H_expr                 # surface elevation (m)
# Flotation thickness: H_f = -(rho_w/rho_i) * b
# The glacier is grounded where H > H_f, i.e. N > 0.
N_expr = rho_i * g * H_expr + rho_w * g * b_expr   # N = p_i + p_w (b<0)
N_min  = Constant(1e3)   # floor: 1 kPa, prevents division by zero
b0 = Function(Q, name="bed").interpolate(b_expr)
h0 = Function(Q, name="thickness").interpolate(H_expr)
s0 = Function(Q, name="surface").interpolate(s_expr)
# Use firedrake.max_value to enforce the floor
N0 = Function(Q, name="N").interpolate(
    firedrake.max_value(N_expr, N_min)
)   # verify against your icepack version
# Quick sanity check: flotation fraction at terminus
Hf_term = -(rho_w / rho_i) * (-100.0)   # flotation thickness at x=L
H_term  = 1000.0 - 300.0               # actual thickness at x=L
print(f"At terminus:  H = {H_term:.0f} m,  H_flotation = {Hf_term:.0f} m")
print(f"  => {'grounded' if H_term > Hf_term else 'FLOATING — check geometry'}")
print(f"  => N_terminus ≈ {rho_i*g*H_term + rho_w*g*(-100):.0f} Pa")

In [ ]:
# --- Visualize the geometry ---
xs = np.linspace(0, L, 400)
b_vals = -100.0 - 600.0 * xs / L
H_vals = 1000.0 - 300.0 * xs / L
s_vals = b_vals + H_vals
N_vals = np.maximum(rho_i * g * H_vals + rho_w * g * b_vals, 1e3)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
ax.fill_between(xs / 1e3, b_vals, s_vals, alpha=0.4, color='steelblue', label='ice')
ax.fill_between(xs / 1e3, b_vals, np.zeros_like(xs), alpha=0.25,
                color='navy', label='ocean (below sea level)')
ax.plot(xs / 1e3, s_vals, 'steelblue', lw=2)
ax.plot(xs / 1e3, b_vals, 'saddlebrown', lw=2, label='bed')
ax.axhline(0, color='navy', lw=1, ls='--', label='sea level')
ax.set(xlabel='x (km)', ylabel='elevation (m)',
       title='Synthetic outlet glacier geometry')
ax.legend(fontsize=8)
ax = axes[1]
ax.plot(xs / 1e3, N_vals / 1e3, color='darkorange', lw=2)
ax.set(xlabel='x (km)', ylabel='N (kPa)',
       title='Effective pressure (full connectivity)')
plt.tight_layout()
plt.show()

## Three friction laws
{doc}`basal-motion` presents three laws in order of increasing physical richness. We implement each as a Python function that returns the magnitude of the basal traction, $\tau_b$, as a function of the sliding speed $u_b$ and (where needed) the effective pressure $N$.
**Law 1 — Weertman power law.**
$$
\tau_b = C\,u_b^{1/m},
$$
with $m$ the inverse of the standard rheological exponent. The bed has no knowledge of $N$: the stress grows without limit as the glacier speeds up. This is the form used in most ensemble projections run before 2015.
**Law 2 — Budd-type (generalized Weertman).**
$$
\tau_b = C\,u_b^{1/m}\,N,
$$
following {cite}`budd1979`. Effective pressure enters multiplicatively, so the bed weakens when the water pressure rises. The stress still grows without bound with speed, so the Iken ceiling {cite}`iken1981` is not enforced.
**Law 3 — Regularized Coulomb.**
$$
\tau_b = \mu N\left(\frac{u_b}{u_b + \lambda N^n}\right)^{1/n},
$$
following {cite}`schoof2005`, {cite}`gagliardini2007`, and {cite}`tsai2015`. At low speed this approaches a Weertman-like power law; at high speed it saturates at the Coulomb ceiling $\mu N$. Because $N \to 0$ near the grounding line, the ceiling itself falls toward zero there — meaning the bed simply cannot supply the traction needed to resist rapid flow, and the grounding line becomes a place of intrinsic weakness.
### Passing a custom friction law to icepack
icepack's hybrid and SSA models accept a `friction` keyword in the `FlowSolver` constructor. The friction argument should be a callable with signature
```python
friction(velocity, thickness, surface, **kwargs)
```
that returns the UFL expression for the friction **power density** (stress times velocity, units of W/m²). icepack uses the action-functional formulation internally, so what you supply is integrated over the bed. See the cells below for the concrete pattern.

In [ ]:
# ----------------------------------------------------------------
# Friction-law parameters
# These are set in the calibration cell below; declared here so
# subsequent cells can reference them.
# ----------------------------------------------------------------
# Shared exponent
m_exp  = 3.0          # Weertman / Budd m  (tau ~ u^(1/m))
n_exp  = 3.0          # Coulomb n
# Law 1: Weertman
C_W = Constant(1.0)   # will be tuned
# Law 2: Budd
C_B = Constant(1.0)   # will be tuned
# Law 3: Regularized Coulomb
mu_C  = Constant(0.5)    # Coulomb coefficient (dimensionless)
lam_C = Constant(1e-2)   # transition parameter (m^(1-n) Pa^(-n))
# Regularization floor for speed (prevents 0/0)
u_min = Constant(1.0)    # 1 m/yr
print("Parameters declared. Run calibration cell before solving.")

In [ ]:
# ----------------------------------------------------------------
# Friction functionals for icepack
# Each returns the power-density integrand psi such that
# the friction action is  A_f = integral(psi dx).
# For a scalar traction f(u), icepack needs psi = int_0^|u| f(s) ds
# (the antiderivative), or you can supply the stress directly if
# using the keyword interface described below.
#
# The simplest modern pattern (icepack >= 1.1) is to supply
#   friction=my_fn  where my_fn(velocity, thickness, surface, **kw)
# returns the traction vector.   # verify against your icepack version
# ----------------------------------------------------------------
from icepack.constants import gravity as g_ip, ice_density as rho_i_ip
def speed(u):
    """Sliding speed with a regularization floor."""
    return firedrake.max_value(sqrt(inner(u, u)), u_min)
def friction_weertman(velocity, thickness, surface, N_field, **kwargs):
    """Law 1: tau_b = C * |u|^(1/m).  No N dependence.
    Returns traction magnitude; icepack multiplies by -u/|u| internally.
    # verify against your icepack version
    """
    ub = speed(velocity)
    tau_mag = C_W * ub ** (1.0 / m_exp)
    return -tau_mag * velocity / ub
def friction_budd(velocity, thickness, surface, N_field, **kwargs):
    """Law 2: tau_b = C * |u|^(1/m) * N.  Budd-type.
    # verify against your icepack version
    """
    ub  = speed(velocity)
    N   = firedrake.max_value(N_field, N_min)
    tau_mag = C_B * ub ** (1.0 / m_exp) * N
    return -tau_mag * velocity / ub
def friction_coulomb(velocity, thickness, surface, N_field, **kwargs):
    """Law 3: regularized Coulomb.  tau_b -> mu*N as u -> inf.
    # verify against your icepack version
    """
    ub = speed(velocity)
    N  = firedrake.max_value(N_field, N_min)
    tau_mag = mu_C * N * (ub / (ub + lam_C * N ** n_exp)) ** (1.0 / n_exp)
    return -tau_mag * velocity / ub
print("Friction functions defined.")

## Setting up the icepack model
We use icepack's `HybridModel` (or `IceStream` for a pure SSA flow) with the lateral boundary conditions as Dirichlet inflow at $x = 0$ and a free calving front at $x = L$. The bed IDs on a `RectangleMesh` are 1 (left, $x=0$), 2 (right, $x=L$), 3 (bottom), 4 (top); we constrain the transverse velocity on ids 3 and 4.
Because we want to swap friction laws without rebuilding the solver each time, we create a small helper `make_solver` and call it three times.

In [ ]:
# --- Initial velocity guess ---
u_inflow = 50.0   # m/yr at x = 0 (inland)
u_init = Function(V, name="u_init").interpolate(
    as_vector((u_inflow + 200.0 * x / L, 0.0))
)
# Temperature and fluidity
T_ice  = Constant(263.0)        # -10 °C, a reasonable outlet value
A_ice  = icepack.rate_factor(T_ice)   # Glen's A
def make_solver(friction_fn):
    """Return a FlowSolver for the IceStream model with the given
    friction function.  Inflow Dirichlet BC at x=0 (boundary id 1);
    free front at x=L.  # verify against your icepack version
    """
    model = icepack.models.IceStream(friction=friction_fn)
    solver = icepack.solvers.FlowSolver(
        model,
        dirichlet_ids=[1],             # inflow
        side_wall_ids=[3, 4],          # no-penetration on walls
    )   # verify against your icepack version
    return model, solver
model_W, solver_W = make_solver(friction_weertman)
model_B, solver_B = make_solver(friction_budd)
model_C, solver_C = make_solver(friction_coulomb)
print("Three solvers created (Weertman, Budd, Coulomb).")

## Calibration: the same speed at the grounding zone from all three laws
This is the key conceptual trap, and it is worth pausing on before running the code.
Every friction law has at least one free coefficient — $C$ for Laws 1 and 2, and $\mu$ for Law 3. In a real inversion, this coefficient is chosen so that the modelled surface speed matches observed GPS or InSAR velocities. The inversion is done at one moment in time, using the present geometry. All three laws can be made to match the same observed speed at the same observed geometry, so the inversion cannot distinguish them.
Here we do the same thing by hand. We run a single diagnostic solve with a provisional set of coefficients, read off the speed near $x = L$ (the grounding zone), and adjust each coefficient until all three laws produce approximately the same speed there — say 250 m/yr. The calibration is approximate; a production study would iterate.
Once calibrated, Laws 1 and 3 agree at the calibration point but will diverge when the geometry changes, because their structural dependence on $N$ is different.

In [ ]:
# ----------------------------------------------------------------
# Helper: run a diagnostic solve and return the velocity field.
# The N_field is passed through kwargs so each friction function
# can pick it up.   # verify against your icepack version
# ----------------------------------------------------------------
def diagnostic(solver, h, s, u_guess, N_field):
    """Thin wrapper around solver.diagnostic_solve."""
    u = Function(V)
    u.assign(u_guess)
    u = solver.diagnostic_solve(
        velocity=u,
        thickness=h,
        surface=s,
        fluidity=A_ice,
        N_field=N_field,           # picked up by the friction functions
    )   # verify against your icepack version
    return u
# --- Initial solve with provisional coefficients ---
# Adjust C_W, C_B, mu_C until the grounding-zone speed is ~250 m/yr
# for all three laws.  The values below are starting points.
C_W.assign(3.5e4)    # Pa (m/yr)^(-1/m)
C_B.assign(8.0e-2)   # Pa (m/yr)^(-1/m) Pa^(-1)
mu_C.assign(0.45)
lam_C.assign(5e-3)
u_W = diagnostic(solver_W, h0, s0, u_init, N0)
u_B = diagnostic(solver_B, h0, s0, u_init, N0)
u_C = diagnostic(solver_C, h0, s0, u_init, N0)
# Extract x-component speed profiles along the centreline (y = Ly/2)
# by interpolating onto a 1-D array
x1d = np.linspace(0, L, 400)
def profile(u_fn, x1d, y_val=0.5e3):
    pts = [(xi, y_val) for xi in x1d]
    return np.array([u_fn.at(p)[0] for p in pts])
spd_W = profile(u_W, x1d)
spd_B = profile(u_B, x1d)
spd_C = profile(u_C, x1d)
print(f"Grounding-zone speed (x = L):")
print(f"  Weertman : {spd_W[-10:].mean():.1f} m/yr")
print(f"  Budd     : {spd_B[-10:].mean():.1f} m/yr")
print(f"  Coulomb  : {spd_C[-10:].mean():.1f} m/yr")
print("Adjust C_W, C_B, mu_C, lam_C above and re-run until these match.")

In [ ]:
# --- Compute and plot tau_b profiles for the baseline geometry ---
def tau_b_weertman(ub_arr):
    return float(C_W) * np.maximum(ub_arr, 1.0) ** (1.0 / m_exp)
def tau_b_budd(ub_arr, N_arr):
    return float(C_B) * np.maximum(ub_arr, 1.0) ** (1.0 / m_exp) * np.maximum(N_arr, 1e3)
def tau_b_coulomb(ub_arr, N_arr):
    N  = np.maximum(N_arr, 1e3)
    ub = np.maximum(ub_arr, 1.0)
    return float(mu_C) * N * (ub / (ub + float(lam_C) * N ** n_exp)) ** (1.0 / n_exp)
N_1d = np.maximum(rho_i * g * (1000.0 - 300.0 * x1d / L)
                  + rho_w * g * (-100.0 - 600.0 * x1d / L), 1e3)
tb_W = tau_b_weertman(spd_W)
tb_B = tau_b_budd(spd_B, N_1d)
tb_C = tau_b_coulomb(spd_C, N_1d)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
ax.plot(x1d / 1e3, spd_W, label='Weertman', lw=2)
ax.plot(x1d / 1e3, spd_B, label='Budd',     lw=2, ls='--')
ax.plot(x1d / 1e3, spd_C, label='Coulomb',  lw=2, ls=':')
ax.set(xlabel='x (km)', ylabel='speed (m/yr)', title='Baseline speed')
ax.legend()
ax = axes[1]
ax.plot(x1d / 1e3, tb_W / 1e3, label='Weertman', lw=2)
ax.plot(x1d / 1e3, tb_B / 1e3, label='Budd',     lw=2, ls='--')
ax.plot(x1d / 1e3, tb_C / 1e3, label='Coulomb',  lw=2, ls=':')
ax.set(xlabel='x (km)', ylabel='tau_b (kPa)', title='Baseline basal traction')
ax.legend()
plt.tight_layout()
plt.show()

## Perturbation experiment: 20 m thinning near the terminus
Warm ocean water melting the base of Thwaites thins the ice near the grounding line. Here we mimic that forcing by subtracting 20 m from the ice thickness over the seaward 50 km of the domain — the region $x \geq 0.75 L$ — and then re-solving diagnostically. The surface elevation drops accordingly.
[Omitted long matching line]
The 20 m thinning is a modest perturbation. At Thwaites the observed thinning rates are 3–8 m/yr, so 20 m represents roughly 3–7 years of forcing [TODO-CITE: Thwaites thinning rate / Joughin et al. or ITGC].

In [ ]:
# --- Thinned geometry ---
dH_thin  = 20.0       # thinning amount, m
x_thin   = 0.75 * L  # thinning starts here (inland edge of forcing region)
# Taper: full thinning for x >= x_thin, zero for x < x_thin
taper = firedrake.conditional(
    firedrake.ge(x, x_thin),
    Constant(1.0),
    Constant(0.0)
)
h_thin = Function(Q, name="h_thin").interpolate(H_expr - dH_thin * taper)
s_thin = Function(Q, name="s_thin").interpolate(b_expr + H_expr - dH_thin * taper)
N_thin = Function(Q, name="N_thin").interpolate(
    firedrake.max_value(
        rho_i * g * (H_expr - dH_thin * taper) + rho_w * g * b_expr,
        N_min
    )
)
print(f"Thinning {dH_thin} m applied for x >= {x_thin/1e3:.0f} km")
# Show how much N drops at the terminus
N_base_term = rho_i * g * 700.0 + rho_w * g * (-100.0)
N_thin_term = rho_i * g * 680.0 + rho_w * g * (-100.0)
print(f"N at terminus: baseline {N_base_term/1e3:.1f} kPa  ->  thinned {N_thin_term/1e3:.1f} kPa")

In [ ]:
# --- Diagnostic solve on thinned geometry ---
u_W2 = diagnostic(solver_W, h_thin, s_thin, u_W, N_thin)
u_B2 = diagnostic(solver_B, h_thin, s_thin, u_B, N_thin)
u_C2 = diagnostic(solver_C, h_thin, s_thin, u_C, N_thin)
spd_W2 = profile(u_W2, x1d)
spd_B2 = profile(u_B2, x1d)
spd_C2 = profile(u_C2, x1d)
# Speed-up ratio at the grounding zone
for name, s0_arr, s1_arr in [('Weertman', spd_W, spd_W2),
                              ('Budd',     spd_B, spd_B2),
                              ('Coulomb',  spd_C, spd_C2)]:
    ratio = s1_arr[-10:].mean() / s0_arr[-10:].mean()
    print(f"{name:8s}  grounding-zone speed: "
          f"{s0_arr[-10:].mean():.1f} -> {s1_arr[-10:].mean():.1f} m/yr  "
          f"(ratio {ratio:.3f})")

In [ ]:
# --- Plot: before / after for speed and tau_b ---
N_thin_1d = np.maximum(
    rho_i * g * (1000.0 - 300.0 * x1d / L - dH_thin * (x1d >= 0.75 * L))
    + rho_w * g * (-100.0 - 600.0 * x1d / L),
    1e3
)
tb_W2 = tau_b_weertman(spd_W2)
tb_B2 = tau_b_budd(spd_B2, N_thin_1d)
tb_C2 = tau_b_coulomb(spd_C2, N_thin_1d)
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True)
data = [
    ('Weertman', spd_W, spd_W2, tb_W, tb_W2, 'C0'),
    ('Budd',     spd_B, spd_B2, tb_B, tb_B2, 'C1'),
    ('Coulomb',  spd_C, spd_C2, tb_C, tb_C2, 'C2'),
]
for j, (name, sp0, sp1, tb0, tb1, col) in enumerate(data):
    ax_spd = axes[0, j]
    ax_spd.plot(x1d / 1e3, sp0, color=col, lw=2, label='baseline')
    ax_spd.plot(x1d / 1e3, sp1, color=col, lw=2, ls='--', label='-20 m')
    ax_spd.set(title=name, ylabel='speed (m/yr)')
    ax_spd.legend(fontsize=8)
    ax_tb = axes[1, j]
    ax_tb.plot(x1d / 1e3, tb0 / 1e3, color=col, lw=2, label='baseline')
    ax_tb.plot(x1d / 1e3, tb1 / 1e3, color=col, lw=2, ls='--', label='-20 m')
    ax_tb.axvline(x_thin / 1e3, color='gray', lw=1, ls=':', label='thinning edge')
    ax_tb.set(xlabel='x (km)', ylabel='tau_b (kPa)')
    ax_tb.legend(fontsize=8)
plt.suptitle('Speed and basal traction: baseline vs. 20 m thinning at terminus',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## Why the Coulomb-capped bed responds more
Look at the speed-up ratios you printed above. The Weertman law gives the smallest speedup because its traction still rises freely with speed — when the driving stress increases, the bed pushes back harder. The Budd law gives a somewhat larger speedup because $N$ enters multiplicatively and falls near the terminus as the ice thins. But the Coulomb law gives the largest speedup of the three.
[Omitted long matching line]
[Omitted long matching line]

## Task 1 — Vary the thinning magnitude
Go back to the thinning cell and change `dH_thin` to 5, 40, and 80 metres. For each value, re-run the two diagnostic solves and fill in the table below.
| Thinning (m) | Weertman speedup | Budd speedup | Coulomb speedup |
|---|---|---|---|
| 5 | | | |
| 20 | | | |
| 40 | | | |
| 80 | | | |
At what thinning does the Coulomb-law glacier begin to show qualitatively different behaviour from the Weertman-law glacier? Can you relate the threshold to the flotation thickness at the terminus?

## Task 2 — Move the thinning inland
Change `x_thin` to `0.50 * L` so that the thinning extends over the seaward half of the domain. How do the speedup ratios change? Now the Budd law's response should be proportionally larger relative to the Weertman law's, because more of the domain has a lower $N$. Does the Coulomb law's advantage increase or decrease?
Think about what this means for the geometry of real marine instability. The retrograde bed means that as the grounding line retreats inland it encounters deeper ice and lower $N$, amplifying the Coulomb-law response. Sketch the feedback loop in words.

## Task 3 — Change the transition parameter $\lambda$
The parameter `lam_C` in the regularized Coulomb law sets the speed at which the transition between the Weertman-like low-speed regime and the Coulomb-saturated high-speed regime occurs. A large $\lambda$ means the transition happens at a lower speed, so for a given flow velocity the law is already near its ceiling. A small $\lambda$ means the law behaves like a Weertman law over a wider range of speeds.
Run the calibration and perturbation experiment with `lam_C` set to `1e-2`, `1e-4`, and `1e-6` (you will need to re-calibrate `mu_C` each time). Plot the three baseline traction profiles on a single panel. As $\lambda \to 0$, in what limit does the regularized Coulomb law reduce to a perfectly plastic bed? How does that limit connect to the till rheology described in {doc}`basal-motion`?

In [ ]:
# Space for Tasks 1-3.  Copy-paste and modify the calibration and
# perturbation cells above as needed.
# YOUR CODE HERE

## Synthesis: which law would you trust to project Thwaites?
The three laws were calibrated to give the same speed today. Their projections diverge the moment the geometry departs from the calibration state. That is the central lesson.
[Omitted long matching line]
[Omitted long matching line]
[Omitted long matching line]
Write a paragraph (here, below the dashed line) addressing the following.
---
1. The three laws were calibrated to agree at one point and one time. Explain in two or three sentences why this calibration is insufficient to select the correct law, and what additional constraint would help discriminate.
2. From your Task 1 results, at what thinning magnitude does the Coulomb-law glacier show a qualitatively different trajectory? Is that thinning plausible for Thwaites over the next decade?
3. In {doc}`../cryosphere/ice-sheets` the marine ice-sheet instability is characterized by a positive feedback between retreat and flux. Which of the three laws admits that feedback most naturally, and why?
---
*(Your answer here.)*

## Optional extension: time-stepping and grounding-line migration
The diagnostic experiments above hold the geometry fixed and compare the instantaneous velocity fields. A fuller picture requires stepping the thickness forward in time with the mass-balance equation while updating the velocity diagnostically at each step, so the grounding line can migrate.
icepack's `prognostic_solve` handles the thickness evolution. The grounding line is implicit in the flotation condition; tracking its position requires checking, at each step, where $H < -(\rho_w/\rho_i) b$ and zeroing the friction there.
If you want to attempt this extension, apply a constant ocean-melt forcing of $\dot{a}_b = -5$ m/yr over the seaward 30 km and integrate forward for 50 years with a 0.1-year time step. Compare the grounding-line retreat distance under each friction law. You should find that the Coulomb-law glacier retreats significantly faster once the grounding line reaches the steeper part of the retrograde bed.
```python
# Sketch of the prognostic loop:
# for step in range(n_steps):
#     u = solver.diagnostic_solve(velocity=u, thickness=h, surface=s,
#                                 fluidity=A_ice, N_field=N, **{})
#     h = solver.prognostic_solve(dt, thickness=h, velocity=u,
#                                 accumulation=a_surface + a_basal)
#     s = firedrake.Function(Q).interpolate(b0 + h)
#     N = ...  # update effective pressure from new h
# verify against your icepack version
```